In [ ]:
import typing
import numpy as np
import matplotlib.pyplot as plt
from math import*
from dataclasses import dataclass


In [ ]:


@dataclass
class Voxel:
    material: float
    origin: tuple

    def __str__(self):
        return f"{self.origin[0]}, {self.origin[1]}, {self.origin[2]}"
    
    def __repr__(self):
        return f"[{self.origin[0]}, {self.origin[1]}, {self.origin[2]}]"


@dataclass
class VocelLineSegment:
    start_point: np.array
    end_point: np.array
    voxel: Voxel


class Grid:

    def __init__(self, dims):
        self.voxels = [[[None for _ in range(dims[0])] for _ in range(dims[1])] for _ in range(dims[2])]

    def add_voxel(self, material: float, origin: tuple):
        voxel = Voxel(material, origin)
        x, y, z = origin
        self.voxels[x][y][z] = voxel



class ParticlePath:

    def __init__(self, p1: np.array, p2: np.array, grid: Grid):
        self.p1 = p1
        self.p2 = p2 
        self.n_vec = p2 - p1 # Un-normadalized normalvector
        # self.p1_vec = np.array(p1)
        # self.p2_vec = np.array(p2)
        self.grid = grid
        self.intersects = []
        self.voxels = []
        self.voxel_line_segments = []


    def point_along_ray_x(self, x):
        y_x = self.p1[1] + ((self.p2[1] - self.p1[1]) / (self.p2[0] - self.p1[0])) * (x - self.p1[0])
        z_x = self.p1[2] + ((self.p2[2] - self.p1[2]) / (self.p2[0] - self.p1[0])) * (x - self.p1[0])
        return np.array([x, y_x, z_x])


    def point_along_ray_y(self, y):
        x_y = self.p1[0] + ((self.p2[0] - self.p1[0]) / (self.p2[1] - self.p1[1])) * (y - self.p1[1])
        z_y = self.p1[2] + ((self.p2[2] - self.p1[2]) / (self.p2[1] - self.p1[1])) * (y - self.p1[1])
        return np.array([x_y, y, z_y])


    def point_along_ray_z(self, z):
        x_z = self.p1[0] + ((self.p2[0] - self.p1[0]) / (self.p2[2] - self.p1[2])) * (z - self.p1[2])
        y_z = self.p1[1] + ((self.p2[1] - self.p1[1]) / (self.p2[2] - self.p1[2])) * (z - self.p1[2])
        return np.array([x_z, y_z, z])
    

    def find_endpoint_voxel(self, p):

        x, y, z = [floor(f) for f in p]
        return self.grid.voxels[x][y][z]


    def find_intersect_voxels(self, intersect):

        x, y, z = [floor(f) for f in intersect[0]]

        if intersect[1] == 0: # x intersect
            return self.grid.voxels[x][y][z] if self.n_vec[0] > 0 else self.grid.voxels[x-1][y][z]
        elif intersect[1] == 1: # y intersect
            return self.grid.voxels[x][y][z] if self.n_vec[1] > 0 else self.grid.voxels[x][y-1][z]
        elif intersect[1] == 2: # z intersect
            return self.grid.voxels[x][y][z] if self.n_vec[2] > 0 else self.grid.voxels[x][y][z-1]



    def decompose_path(self):
        """Decompose path between p1 & p2 into line-segments for each voxel."""

        x_planes = range(ceil(min(self.p1[0], self.p2[0])), floor(max(self.p1[0], self.p2[0]))+1)
        y_planes = range(ceil(min(self.p1[1], self.p2[1])), floor(max(self.p1[1], self.p2[1]))+1)
        z_planes = range(ceil(min(self.p1[2], self.p2[2])), floor(max(self.p1[2], self.p2[2]))+1)


        for x in x_planes:
            p = self.point_along_ray_x(x)
            # print("x", x)
            #print(p)
            if self.planar_constraint(p):
                self.intersects.append([p, 0]) # 0 for x intersect
        
        for y in y_planes:
            p = self.point_along_ray_y(y)
            # print("y", y)
            #print(p)
            if self.planar_constraint(p):
                self.intersects.append([p, 1]) # 1 for y intersect

        for z in z_planes:
            #print("z", z)
            p = self.point_along_ray_z(z)
            if self.planar_constraint(p):
                self.intersects.append([p, 2]) # 2 for z intersect

        self.intersects.sort(key=lambda intersect: (self.p2 - self.p1) @ intersect[0])
        start_voxel = self.find_endpoint_voxel(self.p1)
        self.voxels.append(start_voxel)
        for intersect in self.intersects:
            voxel = self.find_intersect_voxels(intersect)
            self.voxels.append(voxel)

        all_points = [self.p1] + [intersect[0] for intersect in self.intersects] + [self.p2]
        for i in range(len(self.voxels)):
            voxel_line_segment = VocelLineSegment(all_points[i], all_points[i+1], self.voxels[i])
            self.voxel_line_segments.append(voxel_line_segment)


    def planar_constraint(self, p: np.array):
        """Check wether point p along ray lies between p1 & p2."""
        return (self.n_vec @ self.p1 <= self.n_vec @ p <= self.n_vec @ self.p2) 


In [ ]:

def plot_voxel_line_segments(voxel_line_segments, grid_dims=(4, 4)):
    """
    Plots voxel line segments on a 2D grid.
    
    Parameters:
    - voxel_line_segments: list of tuples or objects with start_point and end_point attributes.
    - grid_dims: tuple indicating the size of the grid in (x, y)
    """

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, grid_dims[0])
    ax.set_ylim(0, grid_dims[1])
    ax.set_aspect('equal')

    # Draw grid
    for x in range(grid_dims[0] + 1):
        ax.axvline(x, color='gray', linestyle='--', linewidth=0.5)
    for y in range(grid_dims[1] + 1):
        ax.axhline(y, color='gray', linestyle='--', linewidth=0.5)

    # Plot each line segment in different color
    colors = plt.cm.rainbow(np.linspace(0, 1, len(voxel_line_segments)))
    for i, seg in enumerate(voxel_line_segments):
        # Support both tuple format and object with attributes
        start = seg.start_point[:2] if hasattr(seg, 'start_point') else seg[0][:2]
        end = seg.end_point[:2] if hasattr(seg, 'end_point') else seg[1][:2]
        ax.plot([start[0], end[0]], [start[1], end[1]], color=colors[i], linewidth=2, label=f'Segment {i+1}')
        ax.plot(start[0], start[1], 'ko')  # Start point
        ax.plot(end[0], end[1], 'ks')      # End point

    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title("Voxel Line Segments in 2D Grid")
    plt.grid(False)
    plt.tight_layout()
    plt.show()


In [ ]:
grid = Grid(dims=(1, 4, 4))
material = 1 

for i in range(4):
    for j in range(4):
        grid.add_voxel(material, (i, j, 0))


p1 = np.array([0.5, 3.5, 0.5])
p2 = np.array([3.5, 2.5, 0.5])


path = ParticlePath(p1, p2, grid)
path.decompose_path()


In [ ]:
plot_voxel_line_segments(path.voxel_line_segments)

In [ ]:
c = 3*10**8
water_e_rho = 3.34e23              # [electrons/m^3]
e_charge = 1.602e-19               # [C]
p_e_mass_ratio = 1836.152          # dimensionless
water_I = 74 * e_charge            # [J]
permittivity = 8.854e-12           # [F/m]

proton_m0 = 938.272 # Proton rest mas in terms of MeV

def S_proton(E, rsp=1, I=water_I): 
    return 2 * np.pi * water_e_rho * (p_e_mass_ratio / E) * \
           ((e_charge**2 / (4 * np.pi * permittivity))**2) * np.log(4 * E / (I * p_e_mass_ratio))



def S_proton(E, rsp=1, I=water_I): 
    gamma_sq = (1+(E/proton_m0))**2
    beta_sq = 1-1/gamma_sq

    return 4*np.pi*

    return 2 * np.pi * water_e_rho * (p_e_mass_ratio / E) * \
           ((e_charge**2 / (4 * np.pi * permittivity))**2) * np.log(4 * E / (I * p_e_mass_ratio))

In [ ]:
proton_energy = (100*10**6)*e_charge # 100 MeV

S = stopping_power(proton_energy)

print(S/e_charge)

In [ ]:
c = 3*10**8
water_e_rho = 3.34e23              # [electrons/m^3]
e_charge = 1.602e-19               # [C]
e_mass = 9.109e-31                 # [kg]
e_mass_eV = 0.511                  # [MeV]
water_I = 74                       # [eV]
permittivity = 8.854e-12           # [F/m]

S_constant_SI = (4*np.pi/(e_mass*c**2))*water_e_rho*((e_charge**2)/(4*np.pi*permittivity))**2 # [J/m]
S_constant = 0.001*S_constant_SI/e_charge # [MeV/mm]

def S_proton(E, rsp=1, I_water=74):
    z = 1 
    proton_m0 = 938.272
    return stopping_power(E, rsp, I_water, z, proton_m0)

def stopping_power(E, rsp, I, z, m0):
    gamma_sq = (1+(E/m0))**2
    beta_sq = 1-1/gamma_sq
    return (S_constant/beta_sq) * (np.log(2e6*e_mass_eV*beta_sq*gamma_sq/I) - beta_sq) # [MeV/mm]

print(S_proton(100))
print(S_proton(1))

print(S_constant)

In [ ]:
def water_tank_deterministic(x_0=0, E_0=100, dx=1):

    E = E_0  # MeV
    x = 0    # mm
    E_min = 1

    E_arr = [E]
    x_arr = [x]
    S_arr = []
    
    while E > E_min:
        S = S_proton(E)
        dE = -S*dx
        dx *= (E + dE)/E
        E += dE
        x += dx
        E_arr.append(E)
        x_arr.append(x)
        S_arr.append(S)

    return np.array(x_arr), np.array(E_arr), np.array(S_arr)

In [ ]:
def water_tank_stochastic(x_0=0, E_0=100, dx=1):

    E = E_0  # MeV
    x = 0    # mm
    E_min = 1

    E_arr = [E]
    x_arr = [x]
    
    while E > E_min:
        S = S_proton(E)
        #S_rand = np.random.normal(S, np.sqrt(S))
        dE = S*dx
        dE_rand = np.random.normal(dE, np.sqrt(dE))
        dx *= (E - dE)/E
        E += dE
        x += dx
        E_arr.append(E)
        x_arr.append(x)

    return np.array(x_arr), np.array(E_arr)

In [ ]:


x_arrs = []
S_arrs = []

n_prim = 10000

for i in range(n_prim):
    E_init = np.random.normal(100, 2)
    x_det, E_det = water_tank_stochastic(dx=1, E_0=E_init)
    x_arrs.append(x_det)



fig, ax = plt.subplots()
ax.scatter(x_det, E_det)
plt.show()

print(E_det[-1])
print(x_det[-1]-x_det[-2])
print(len(x_det))


# depth, dose = calc_dose(x_arrs, S_arrs, dx=1)



In [ ]:
fig, ax = plt.subplots()

ax.plot(depth, dose)

In [ ]:
def calc_dose(x_arrs, S_arrs, dx, x_max=120):
    
    dose = np.zeros(x_max+1)
    n_histories = len(x_arrs)
    for i in range(n_histories):
        for j in range(len(x_arrs[i])-1):
            dose_idx = floor(x_arrs[i][j])
            dose[dose_idx] += S_arrs[i][j]
    
    depth = [i for i in range(x_max+1)]
    return depth, dose/n_histories


